# Resource Selection Function

In [1]:
import pandas as pd
import geopandas as gpd
import xarray as xr
import numpy as np
import ee

from movement_models import (
    FeatureSpec,
    _build_design_matrix,
    _get_availability_domain,
    _get_sampling_points,
    _sample_env_layer,
    load_presence_csvs,
    to_reloc_gdf,
    reproject_reloc,
    to_reloc_gdf_projected,
    init_ee,
    init_ee_on_client,
    make_aoi,
    aoi_to_ee,
    build_predictors_image,
    ee_image_to_env_xarray,
    save_env_zarr,
    load_env_zarr,
    shutdown_default_client,
    make_local_dask_client,
    
    fit_rsf,
    predict_rsf_points,
    get_rsf_surface,
    fixed_width_Boyce,
    sliding_window_Boyce,
    cv_model,
    eval_all_linear_candidates
)

In [16]:
def _choose_chunk_size_points(env, *, client=None, frac_of_worker_mem=0.10, per_worker_budget_mb=None, k=10,
    min_points=1_000, max_points=200_000):

    if client is not None:
        info = client.scheduler_info()["workers"]
        limits_mb = [w["memory_limit"] / 1024**2 for w in info.values()]
        base_budget_mb = min(limits_mb) * frac_of_worker_mem
        per_worker_budget_mb = int(max(8, min(512, base_budget_mb)))
    else:
        if per_worker_budget_mb is None:
            per_worker_budget_mb = 16  # conservative fallback
        per_worker_budget_mb = int(per_worker_budget_mb)

    if hasattr(env, "dims") and "band" in env.dims:
        n_bands = int(env.sizes["band"])
        dtype = env.dtype
    else:
        n_bands = len(env.data_vars)
        dtype = next(iter(env.data_vars.values())).dtype

    bytes_per_value = np.dtype(dtype).itemsize
    budget_bytes = per_worker_budget_mb * 1024 * 1024

    bytes_per_point = n_bands * bytes_per_value
    n_points = budget_bytes // (k * bytes_per_point)

    return int(min(max_points, max(min_points, n_points)))

In [17]:
def _sample_env_layer_old(
    samples: gpd.GeoDataFrame,
    env: xr.DataArray,
    chunk_size_points="auto",
    *,
    client=None,
    per_worker_budget_mb=16,
    k=10,
    min_points=1_000,
    max_points=200_000,
):
    try:
        env_crs = env.rio.crs
    except Exception:
        env_crs = None

    if env_crs is None:
        raise ValueError("env has no rio CRS; set it with env = env.rio.write_crs('EPSG:...')")

    if samples.crs is None:
        raise ValueError("samples.crs is None; set a CRS on your GeoDataFrame before sampling")

    if samples.crs != env_crs:
        samples = samples.to_crs(env_crs)

    if chunk_size_points == "auto" or chunk_size_points is None:
        chunk_size_points = _choose_chunk_size_points(
            env,
            client=client,
            per_worker_budget_mb=per_worker_budget_mb,
            k=k,
            min_points=min_points,
            max_points=max_points,
        )
    else:
        chunk_size_points = int(chunk_size_points)

    xs = xr.DataArray(samples.geometry.x.to_numpy(), dims="points", name="x")
    ys = xr.DataArray(samples.geometry.y.to_numpy(), dims="points", name="y")

    sampled = env.sel(x=xs, y=ys, method="nearest").chunk({"points": chunk_size_points})

    arr = sampled.data
    arr = arr.compute() if hasattr(arr, "compute") else np.asarray(arr)
    arr = np.asarray(arr, dtype="float32")

    bands = sampled["band"].to_numpy().tolist()
    df = pd.DataFrame(arr.T, columns=bands)
    df["x"] = xs.to_numpy()
    df["y"] = ys.to_numpy()
    df["used"] = samples["used"].to_numpy()
    df["Timestamp"] = samples["Timestamp"].to_numpy()
    return df

In [3]:
import importlib
import movement_models.sampling as smp
importlib.reload(smp)

_sample_env_layer_new = smp._sample_env_layer

In [5]:
subset_predictors = ["ndvi", "ndwi", "slope", "dist2water"]
cov = env.sel(band=subset_predictors)

domain = _get_availability_domain(reloc)
samples = _get_sampling_points(domain, 10_000, df=reloc.iloc[:1000], seed=1)

/Users/lubovkabo/Documents/TUM/MolZoologyLab/KONWIHR/HSA/movement_models/sampling.py:45: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  samples["used"] = samples["used"].fillna(False)


In [18]:
from time import perf_counter

t0 = perf_counter()
out_old = _sample_env_layer_old(samples, cov, chunk_size_points="auto", client=client)
t1 = perf_counter()

t2 = perf_counter()
out_new = _sample_env_layer(samples, cov, chunk_size_points="auto", client=client)
t3 = perf_counter()

print("old runtime:", t1 - t0)
print("new runtime:", t3 - t2)

old runtime: 161.1636122079799
new runtime: 48.17169458302669


In [13]:
print(out_old.shape, out_new.shape)
print(list(out_old.columns))
print(list(out_new.columns))

(11000, 8) (11000, 8)
['ndvi', 'ndwi', 'slope', 'dist2water', 'x', 'y', 'used', 'Timestamp']
['ndvi', 'ndwi', 'slope', 'dist2water', 'x', 'y', 'used', 'Timestamp']


In [14]:
import numpy as np

sort_cols = ["Timestamp", "used", "x", "y"]

a = out_old.sort_values(sort_cols).reset_index(drop=True)
b = out_new.sort_values(sort_cols).reset_index(drop=True)

print((a["used"].values == b["used"].values).all())
print((a["Timestamp"].values == b["Timestamp"].values).all())

float_cols = ["x", "y"] + subset_predictors

for col in float_cols:
    ok = np.allclose(a[col].to_numpy(), b[col].to_numpy(), equal_nan=True, rtol=1e-7, atol=1e-8)
    max_diff = np.nanmax(np.abs(a[col].to_numpy() - b[col].to_numpy()))
    print(col, ok, max_diff)

True
False
x True 0.0
y True 0.0
ndvi True 0.0
ndwi True 0.0
slope True 0.0
dist2water True 0.0


In [15]:
spec = FeatureSpec(linear=subset_predictors, add_const=True)

m_old, scaler_old, _ = fit_rsf(out_old, spec)
pred_old = predict_rsf_points(out_old, m_old, scaler_old, spec)
rsf_old = get_rsf_surface(cov, m_old, scaler_old, spec, crs=cov.rio.crs)
B_old, _ = fixed_width_Boyce(pred_old, rsf_old, domain, seed=1)

m_new, scaler_new, _ = fit_rsf(out_new, spec)
pred_new = predict_rsf_points(out_new, m_new, scaler_new, spec)
rsf_new = get_rsf_surface(cov, m_new, scaler_new, spec, crs=cov.rio.crs)
B_new, _ = fixed_width_Boyce(pred_new, rsf_new, domain, seed=1)

print("Boyce old:", B_old)
print("Boyce new:", B_new)

Boyce old: 0.7022556390977444
Boyce new: 0.7022556390977444


In [2]:
client = make_local_dask_client(memory_limit="2GB") 

init_status = init_ee_on_client(client, project_id="ee-alvykabo")
client, init_status

(<Client: 'tcp://127.0.0.1:50760' processes=5 threads=5, memory=9.31 GiB>,
 {'tcp://127.0.0.1:50773': 1,
  'tcp://127.0.0.1:50776': 1,
  'tcp://127.0.0.1:50777': 1,
  'tcp://127.0.0.1:50778': 1,
  'tcp://127.0.0.1:50785': 1})

2026-03-21 17:24:22,073 - tornado.application - ERROR - Exception in callback <bound method SystemMonitor.update of <SystemMonitor: cpu: 7 memory: 159 MB fds: 246>>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/cleanjupyter/lib/python3.12/site-packages/tornado/ioloop.py", line 945, in _run
    val = self.callback()
          ^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/cleanjupyter/lib/python3.12/site-packages/distributed/system_monitor.py", line 168, in update
    net_ioc = psutil.net_io_counters()
              ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/cleanjupyter/lib/python3.12/site-packages/psutil/__init__.py", line 2148, in net_io_counters
    rawdict = _psplatform.net_io_counters()
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: [Errno 12] Cannot allocate memory


## 1. Preparation

### 1.1 Fetch presence data

In [19]:
reloc_df = load_presence_csvs("data/*.csv")
reloc = to_reloc_gdf(reloc_df)

reloc.head(), reloc.crs, reloc["Timestamp"].dtype

(            Timestamp     ID  Latitude  Longitude                    geometry
 0 2024-05-04 14:31:32  NPL38 -19.50359   14.21912  POINT (14.21912 -19.50359)
 1 2024-05-04 14:32:42  NPL38 -19.50360   14.21912   POINT (14.21912 -19.5036)
 2 2024-05-04 20:37:42  NPL38 -19.51765   14.21877  POINT (14.21877 -19.51765)
 3 2024-05-04 20:38:09  NPL38 -19.51763   14.21878  POINT (14.21878 -19.51763)
 4 2024-05-04 22:43:10  NPL38 -19.47570   14.22619   POINT (14.22619 -19.4757),
 <Geographic 2D CRS: EPSG:4326>
 Name: WGS 84
 Axis Info [ellipsoidal]:
 - Lat[north]: Geodetic latitude (degree)
 - Lon[east]: Geodetic longitude (degree)
 Area of Use:
 - name: World.
 - bounds: (-180.0, -90.0, 180.0, 90.0)
 Datum: World Geodetic System 1984 ensemble
 - Ellipsoid: WGS 84
 - Prime Meridian: Greenwich,
 dtype('<M8[ns]'))

### 1.2 Fetch environmental layers

In [20]:
ee.Authenticate()
init_ee(project_id="ee-alvykabo")   

aoi = make_aoi(reloc, buffer_m=1)
aoi_ee = aoi_to_ee(aoi)

predictors_img = build_predictors_image(aoi_ee, start="2024-01-01", end="2024-12-31")
env = ee_image_to_env_xarray(predictors_img, aoi_ee, crs="EPSG:29333", scale=100, chunk_xy=1024)

save_env_zarr(env, "env_29333.zarr", mode="w")
env2 = load_env_zarr("env_29333.zarr")

env.shape, env.rio.crs, env2.shape

/opt/anaconda3/envs/cleanjupyter/lib/python3.12/site-packages/xee/ext.py:696: UserWarning: Unable to retrieve 'system:time_start' values from an ImageCollection due to: No 'system:time_start' values found in the 'ImageCollection'.
  warnings.warn(
/opt/anaconda3/envs/cleanjupyter/lib/python3.12/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


((4, 2663, 2310),
 CRS.from_wkt('PROJCS["Schwarzeck / UTM zone 33S",GEOGCS["Schwarzeck",DATUM["Schwarzeck",SPHEROID["Bessel Namibia (GLM)",6377483.86528042,299.1528128,AUTHORITY["EPSG","7046"]],AUTHORITY["EPSG","6293"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4293"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",15],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",10000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","29333"]]'),
 (4, 2663, 2310))

## 2. Calculate the RSF

### 2.1 Define the sampling scheme

In [21]:
domain = _get_availability_domain(reloc)
samples = _get_sampling_points(domain, 1000, df=reloc.iloc[:100], seed=1)
sampled = _sample_env_layer(samples, env.sel(band=["ndvi", "slope"]), chunk_size_points="auto", client=client)

sampled.head()

/Users/lubovkabo/Documents/TUM/MolZoologyLab/KONWIHR/HSA/movement_models/sampling.py:45: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  samples["used"] = samples["used"].fillna(False)


,ndvi,slope,x,y,used,Timestamp
0,0.108499,22.150867,418126.303254,7.843514e+06,True,2024-05-04 14:31:32
1,0.108499,22.150867,418126.308290,7.843513e+06,True,2024-05-04 14:32:42
2,0.089272,3.929971,418096.660845,7.841958e+06,True,2024-05-04 20:37:42
3,0.089272,3.929971,418097.700044,7.841961e+06,True,2024-05-04 20:38:09
4,0.136155,0.463735,418854.301550,7.846604e+06,True,2024-05-04 22:43:10


### 2.2 Extract values from the environmental covariates

### 2.3 Fit logistic regression

In [22]:
spec = FeatureSpec(linear=["ndvi", "slope"], add_const=True)
m, scaler, _ = fit_rsf(sampled, spec)
pred = predict_rsf_points(sampled, m, scaler, spec)

pred[["rsf_pred"]].describe()

,rsf_pred
count,1100.000000
mean,0.105094
std,0.096207
min,0.017897
25%,0.072091
50%,0.087939
75%,0.111392
max,1.681505


### 2.4 Get RSF surface

In [23]:
rsf = get_rsf_surface(env.sel(band=spec.linear), m, scaler, spec)

rsf

<xarray.DataArray 'rsf' (band: 1, y: 2663, x: 2310)> Size: 25MB
dask.array<getitem, shape=(1, 2663, 2310), dtype=float32, chunksize=(1, 1024, 1024), chunktype=numpy.ndarray>
Coordinates:
  * band         (band) object 8B 'rsf'
  * y            (y) float64 21kB 7.702e+06 7.702e+06 ... 7.968e+06 7.968e+06
  * x            (x) float64 18kB 3.002e+05 3.003e+05 ... 5.31e+05 5.311e+05
    spatial_ref  int64 8B 0

In [24]:
rsf = get_rsf_surface(env, m, scaler, spec)
rsf = rsf.compute()
rsf.rio.to_raster("rsf.tif", compress="LZW")
rsf = rsf.rio.write_crs("EPSG:29333") 

In [35]:
client.scheduler_info()["workers"].keys()
len(client.scheduler_info()["workers"])

5

In [36]:
import pandas as pd

info = client.scheduler_info()["workers"]

df = pd.DataFrame({
    w: {
        "memory": data["metrics"]["memory"],
        "memory_limit": data["memory_limit"]
    }
    for w, data in info.items()
}).T

df

,memory,memory_limit
tcp://127.0.0.1:50773,429916160,2000000000
tcp://127.0.0.1:50776,353386496,2000000000
tcp://127.0.0.1:50777,360529920,2000000000
tcp://127.0.0.1:50778,346669056,2000000000
tcp://127.0.0.1:50785,323616768,2000000000


In [37]:
df["memory_MB"] = df["memory"] / 1e6
df["limit_MB"] = df["memory_limit"] / 1e6
df[["memory_MB", "limit_MB"]]

,memory_MB,limit_MB
tcp://127.0.0.1:50773,429.916160,2000.0
tcp://127.0.0.1:50776,353.386496,2000.0
tcp://127.0.0.1:50777,360.529920,2000.0
tcp://127.0.0.1:50778,346.669056,2000.0
tcp://127.0.0.1:50785,323.616768,2000.0


### 2.5 Evaluate Using Boyce Index

In [31]:
subset_predictors = ["ndvi", "ndwi", "slope", "dist2water"]
cov = env.sel(band=subset_predictors)

samples = _get_sampling_points(domain, n=10_000, df=reloc.iloc[:1000], seed=1)
sampled = _sample_env_layer(samples, cov, chunk_size_points="auto", client=client)

spec = FeatureSpec(linear=subset_predictors, add_const=True)
m, scaler, spec = fit_rsf(sampled, spec)

pred = predict_rsf_points(sampled, m, scaler, spec)
rsf = get_rsf_surface(env.sel(band=subset_predictors), m, scaler, spec)

B, boyce = fixed_width_Boyce(pred, rsf, domain, seed=1)
B, boyce.head()

/Users/lubovkabo/Documents/TUM/MolZoologyLab/KONWIHR/HSA/movement_models/sampling.py:45: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  samples["used"] = samples["used"].fillna(False)


(0.7022556390977444,
    q_mid   rsf_mid        pe
 0  0.025 -3.860504  0.420084
 1  0.075 -3.521645  0.540000
 2  0.125 -3.383368  0.200040
 3  0.175 -3.272449  0.199960
 4  0.225 -3.167086  0.160000)

In [32]:
B, chart = sliding_window_Boyce(
    pred=pred,
    rsf=rsf,
    domain=domain,
    window_frac=0.2,
    step_frac=0.05,
    seed=1,
)

print("B =", B)
print(chart.head())
print("rows:", len(chart), "finite pe:", chart["pe"].notna().sum())

B = 0.7941176470588236
    rsf_mid        pe
0 -3.672495  0.340000
1 -3.359937  0.280000
2 -3.220671  0.145026
3 -3.095796  0.095000
4 -2.974046  0.050002
rows: 17 finite pe: 17


## 3. Aggregate functions

### 3.1 Compare all possible linear models via AIC & BIC

In [33]:
df_small = sampled.sample(n=min(2000, len(sampled)), random_state=1).copy()
subset_predictors = ["ndvi", "ndwi", "slope", "dist2water"]

df_small = df_small.replace([float("inf"), float("-inf")], pd.NA).dropna(subset=["used"] + subset_predictors)

res = eval_all_linear_candidates(df_small, env, subset=subset_predictors)
res.head()

,Variables,AIC,BIC
0,[ndvi],1281.569335,1292.771140
1,[ndwi],1279.145729,1290.347534
2,[slope],1276.215558,1287.417363
3,[dist2water],1221.289344,1232.491149
4,"[ndvi, ndwi]",1281.084730,1297.887437


### 3.2 Cross-Validation on a Single Individual

In [34]:
res = cv_model(
    obs=reloc,
    env=env,
    k_folds=2,
    subset_predictors=["ndvi", "ndwi", "slope", "dist2water"],
    sampling_factor_train=3,
    n_bg_boyce=5_000,
    seed=1,
)
res

/Users/lubovkabo/Documents/TUM/MolZoologyLab/KONWIHR/HSA/movement_models/rsf/cv.py:33: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  min_dt = pd.Timedelta(min_dt)
/Users/lubovkabo/Documents/TUM/MolZoologyLab/KONWIHR/HSA/movement_models/sampling.py:45: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  samples["used"] = samples["used"].fillna(False)
/Users/lubovkabo/Documents/TUM/MolZoologyLab/KONWIHR/HSA/movement_models/rsf/cv.py:33: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  min_dt = pd.Timedelta(min_dt)
/Users/lubovkabo/Documents/TUM/MolZoologyLab/KONWIHR/HSA/movement_models/sampling.py:45: FutureWarning: Downcasting object dtype arrays on .fillna, .

,k,boyce
0,0,NaN
1,1,NaN


# Step, Length, & Turning Angle

In [38]:
#testing movement_kernel.py with likelihood-based fitting

from movement_models.movement import (
    prepare_trajectory_data,
    build_step_data,
    fit_step_distribution,
    build_turn_angle_data,
    fit_turn_angle_distribution,
    fit_movement_kernel_per_id,
)

from movement_models.io import (
    load_presence_csvs,
    to_reloc_gdf_projected,
)

from movement_models.io import load_presence_csvs
from movement_models.movement import fit_movement_kernel_per_id

reloc = load_presence_csvs("data/*.csv")

out = fit_movement_kernel_per_id(
    reloc,
    id_col="ID",
    timestamp_col="Timestamp",
    lon_col="Longitude",
    lat_col="Latitude",
    target_crs="EPSG:29333",
    expected_interval_min=120,
    tolerance_min=2,
    step_cutoff=20000,
)

out["summary"]

,ID,n_steps,step_distribution,step_params,step_q25,step_median,step_mean,step_q75,step_max,n_angles,vonmises_kappa,mixture_kappa,mixture_w
0,NPL38,2584,lognorm,"[2.3680072661020657, 76.329595410113]",8.809685,63.692252,546.335874,760.120739,7014.598776,2252,0.135231,6.79407,0.103583


In [41]:
step_df = out["step_df"]
one_id = step_df["ID"].dropna().iloc[0]
step_subset = step_df.loc[step_df["ID"] == one_id, "step_m"]

fit_step_distribution(step_subset)["model_table"]

,distribution,params,loglik,AIC,success
0,lognorm,"[2.3680072661020657, 76.329595410113]",-17095.920956,34195.841913,True
1,weibull,"[0.4683982286268443, 249.18223047700394]",-17177.628049,34359.256097,True
2,gamma,"[0.3428314692397524, 1593.5262762864115]",-17250.213319,34504.426638,True
3,exp,[546.3358741887124],-18871.556503,37745.113006,True
